<a href="https://colab.research.google.com/github/retogo/llm2026/blob/main/%E5%A4%A7%E8%A6%8F%E6%A8%A1%E8%A8%80%E8%AA%9E%E3%83%A2%E3%83%86%E3%82%99%E3%83%AB1_%E7%AC%AC7%E5%9B%9E%E6%BC%94%E7%BF%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 大規模言語モデル1 第7回演習
## 演習の目的
講義パートでは強化学習の目的やアルゴリズムについて説明を行いました

本演習では、強化学習を用いてLLMを学習する流れを体験していただきます。


### ライブラリのインストール

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
if "COLAB_" not in "".join(os.environ.keys()):
    # Google Colabでない場合はpip installを行う
    !pip install unsloth vllm
    !pip install mecab-python3 unidic-lite
else:
    pass

In [ ]:
#@title Google Colab用の追加インストール { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
  # Google Colabでない場合はpip installを行う
    !pip install unsloth vllm
else:
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao==0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install mecab-python3 unidic-lite

In [ ]:
from unsloth import FastModel
import torch

# モデルの最大の出力長
max_seq_length = 1024

# モデルの読み込み
# 今回はGemma3 1Bを用いる
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,

    # Flex AttentionのHalf/Float不整合を回避
    attn_implementation = "eager",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:226: UserWarning: torchcodec 0.11.0+cu128 is incompatible with torch 2.7.0+cu126; install a matching build with `pip install 'torchcodec>=0.5,<0.6.0'`.
  disable_torchcodec_if_broken()


INFO 08-06 07:55:37 [__init__.py:244] Automatically detected platform cuda.
ERROR 08-06 07:55:42 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.5: Fast Gemma3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


LoRAを用いて学習(LoRAについては第6回を復習)

In [ ]:
# LoRAを学習するための準備
# rank=8で学習
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

### データ準備
<a name="Data"></a>

今回はLLMで短歌を出力できるように学習を行う。

短歌のテーマ一覧をWAON-Benchデータセットから取得する。

※ 本来WAON-Benchデータセットは日本文化におけるVision-Languageモデルのベンチマーク評価を目的として設計された、手動でキュレーションされた画像分類データセットであるが、今回はclassラベルを用いる。

In [ ]:
from datasets import load_dataset
dataset = load_dataset("llm-jp/WAON-Bench", split = "train")
dataset

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

data/train-00004-of-00032.parquet:   0%|          | 0.00/5.84k [00:00<?, ?B/s]

data/train-00001-of-00032.parquet:   0%|          | 0.00/5.00k [00:00<?, ?B/s]

...

data/train-00029-of-00032.parquet:   0%|          | 0.00/5.43k [00:00<?, ?B/s]

data/train-00030-of-00032.parquet:   0%|          | 0.00/5.47k [00:00<?, ?B/s]

data/train-00031-of-00032.parquet:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1870 [00:00<?, ? examples/s]

Dataset({
    features: ['class', 'url', 'category'],
    num_rows: 1870
})

In [ ]:
# クラスラベルの重複を削除
seen = set()
dataset = dataset.filter(lambda x: not (x["class"] in seen or seen.add(x["class"])))

print(dataset[0]['class'])
print(dataset[1]['class'])

Filter:   0%|          | 0/1870 [00:00<?, ? examples/s]

柴犬
秋田犬


システムプロンプトの設定

In [ ]:
reasoning_start = "<思考>"
reasoning_end   = "</思考>"
solution_start = "<答え>"
solution_end = "</答え>"

system_prompt = \
f"""短歌を作るためのテーマが1つ与えられます。
{reasoning_start} と {reasoning_end} の間に、短歌を作るための思考過程や推敲の過程を記述してください。
特に5-7-5-7-7の形式に従っているかstep-by-stepで確認し、必要に応じて修正を加えてください。
その後、完成した短歌を、{solution_start} と {solution_end} の間に 5-7-5-7-7 の形式で一首だけ出力してください。
## 例:
{reasoning_start}
テーマから考えると、春の情景を描くのが良いでしょう。
まず、5-7-5の部分を考えます。
「ひさかたの」（5モーラ）
「光のどけき」（7モーラ）
「春の日に」（5モーラ）
次に、7-7の部分を考えます。
「しづ心なく」（7モーラ）
「花の散るらむ」（7モーラ）
全体で5-7-5-7-7の形式に従っています。
したがって、以下の短歌を完成させます。
{reasoning_end}
{solution_start}
ひさかたの
光のどけき
春の日に
しづ心なく
花の散るらむ
{solution_end}"""

print(system_prompt)

短歌を作るためのテーマが1つ与えられます。
<思考> と </思考> の間に、短歌を作るための思考過程や推敲の過程を記述してください。
特に5-7-5-7-7の形式に従っているかstep-by-stepで確認し、必要に応じて修正を加えてください。
その後、完成した短歌を、<答え> と </答え> の間に 5-7-5-7-7 の形式で一首だけ出力してください。
## 例:
<思考>
テーマから考えると、春の情景を描くのが良いでしょう。
まず、5-7-5の部分を考えます。
「ひさかたの」（5モーラ）
「光のどけき」（7モーラ）
「春の日に」（5モーラ）
次に、7-7の部分を考えます。
「しづ心なく」（7モーラ）
「花の散るらむ」（7モーラ）
全体で5-7-5-7-7の形式に従っています。
したがって、以下の短歌を完成させます。
</思考>
<答え>
ひさかたの
光のどけき
春の日に
しづ心なく
花の散るらむ
</答え>


学習用のデータセットの準備

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["class"]},
    ],
})
dataset[0]

Map:   0%|          | 0/374 [00:00<?, ? examples/s]

{'class': '柴犬',
 'url': 'https://img.wanqol.com/2020/11/6e489894-main.jpg?auto=format',
 'category': 'animal',
 'prompt': [{'role': 'system',
   'content': '短歌を作るためのテーマが1つ与えられます。\n<思考> と </思考> の間に、短歌を作るための思考過程や推敲の過程を記述してください。\n特に5-7-5-7-7の形式に従っているかstep-by-stepで確認し、必要に応じて修正を加えてください。\nその後、完成した短歌を、<答え> と </答え> の間に 5-7-5-7-7 の形式で一首だけ出力してください。\n## 例:\n<思考>\nテーマから考えると、春の情景を描くのが良いでしょう。\nまず、5-7-5の部分を考えます。\n「ひさかたの」（5モーラ）\n「光のどけき」（7モーラ）\n「春の日に」（5モーラ）\n次に、7-7の部分を考えます。\n「しづ心なく」（7モーラ）\n「花の散るらむ」（7モーラ）\n全体で5-7-5-7-7の形式に従っています。\nしたがって、以下の短歌を完成させます。\n</思考>\n<答え>\nひさかたの\n光のどけき\n春の日に\nしづ心なく\n花の散るらむ\n</答え>'},
  {'role': 'user', 'content': '柴犬'}]}

### 報酬関数の設計
強化学習を行う上で重要なのが報酬関数をどのように設計するかです。
この設計が不適切だと全く学習することができません。

例えば数学能力を向上させる場合は、答えが正解しているかどうか、コーディング能力を向上させる場合は、テストに通るかどうかが報酬となります。

今回は、短歌を生成したいので、生成した文章が5, 7, 5, 7, 7にしたがっているかどうかを報酬とします。

また、出力が先ほど設定したフォーマットに正しく従っているかも報酬として設計します。

※ 今回はシンプルな報酬として形式に従っているかだけを報酬としているので、短歌の質を報酬としているわけでないことに注意してください

In [ ]:
import MeCab

# 定数
TANKA_PATTERN = [5, 7, 5, 7, 7]
SMALL_KANA = set("ゃゅょぁぃぅぇぉゎァィゥェォャュョヮ")
SPECIAL_MORA = set("っッんンー")

mecab = MeCab.Tagger()

# カタカナをひらがなに変換
def kata_to_hira(text):
    return "".join(
        chr(ord(ch) - 0x60) if 0x30A1 <= ord(ch) <= 0x30F6 else ch
        for ch in text
    )

# 漢字を含む日本語テキストをひらがな読みに変換
def text_to_hiragana(text):
    node = mecab.parseToNode(text)
    result = []

    while node:
        features = node.feature.split(",")
        if len(features) > 9 and features[9] != "*":
            result.append(kata_to_hira(features[9]))
        else:
            result.append(node.surface)
        node = node.next

    return "".join(result)

# ひらがな文字列のモーラ数をカウント
def count_mora(hiragana):
    mora_count = 0
    prev_is_normal = False

    for char in hiragana:
        if char in SPECIAL_MORA:
            mora_count += 1
            prev_is_normal = False
        elif char in SMALL_KANA and prev_is_normal:
            prev_is_normal = False
        elif "ぁ" <= char <= "ん":
            mora_count += 1
            prev_is_normal = True
        else:
            prev_is_normal = False

    return mora_count


# 短歌が5-7-5-7-7に従っているかをスコアリング
# スコア計算:
# - 完全一致: +1.0点
# - ±1モーラ: +0.5点
# - 行数が5でない: -1.0点
def score_tanka(text):
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]

    score = 0.0
    if len(lines) != 5:
        score -= 1.0

    for i in range(min(5, len(lines))):
        hiragana = text_to_hiragana(lines[i])
        mora_count = count_mora(hiragana)
        difference = abs(mora_count - TANKA_PATTERN[i])

        if difference == 0:
            score += 1.0
        elif difference == 1:
            score += 0.5

    return score

# 出力をチェックして答えの中身の報酬を計算
def check_answer(prompts, completions, **kwargs):
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess in extracted_responses:
        if guess is None:
            scores.append(0)
            continue
        score = score_tanka(guess)
        scores.append(score)
    return scores


短歌の報酬が正しく設計できているかのチェック

In [ ]:
sample = """
ひさかたの
光のどけき
春の日に
しづ心なく
花の散るらむ
"""
score_tanka(sample)

5.0

次に正しくフォーマットに従っているかも報酬として設計します。

In [ ]:
import re

# フォーマットの正規表現
match_format = re.compile(
    rf"^[\s]{{0,}}"\
    rf"{reasoning_start}.+?{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)

match_format.search(
    "<思考>考え中...</思考>"\
    "<答え>ここで短歌を出力</答え>",
)

<re.Match object; span=(0, 32), match='<思考>考え中...</思考><答え>ここで短歌を出力</答え>'>

フォーマットに完全一致した時は3ポイントの報酬を与えます

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

もし完全一致しなかった場合は，それぞれ<思考></思考><答え></答え>のトークンがちょうど1回だけ出力された場合に0.5ポイントの報酬を与えます

In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        score += 0.5 if response.count(reasoning_start) == 1 else -0.5
        score += 0.5 if response.count(reasoning_end)   == 1 else -0.5
        score += 0.5 if response.count(solution_start)  == 1 else -0.5
        score += 0.5 if response.count(solution_end)    == 1 else -0.5
        scores.append(score)
    return scores

<a name="Train"></a>
### モデルの学習

GRPO Trainerを用いて学習

学習前の状態の出力を確認

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "秋の空"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
)
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<思考>
テーマ：秋の空。
「秋の空」という言葉から連想されるのは、夕焼けの美しさ、静けさ、そして少し寂しさを感じる感覚。色々な表現を思い浮かべましたが、5-7-7のパターンに沿って、空の姿を表現する、少し寂しげで美しい短歌にしたいと思いました。

まずは「空」という言葉から、少し遠くの空の広がりを表現してみようと考えました。そして、夕焼けというイメージを合わせることで、空の美しさを表現したいです。5-7-7のルールに沿って、少し寂しげな感情を込めて表現したいですね。

いくつか候補が浮かびました。

1.  **空の静けさ**：空の静けさ、寂しさを表現する
2.  **茜色の空**：夕焼けの茜色を表現する
3.  **遠い空**：遠い空の空の広がり

これらの候補の中で、特に「茜色の空」の方が、夕焼けの美しさと寂しさを組み合わせるのに適していると感じました。

以下に短歌を作成します。

茜色の空
遠い空の静けさ
秋の空の夢
風の音を聴う
</思考>
<答え>
茜色の空
遠い空の静けさ
秋の空の夢
風の音を聴う
</答え><end_of_turn>


In [ ]:
from trl import GRPOConfig, GRPOTrainer

# promptの最大長
max_prompt_length = 256

# GRPOの学習設定
training_args = GRPOConfig(
    learning_rate = 5e-6,                 # 学習率
    adam_beta1 = 0.9,                     # Adamのハイパラ
    adam_beta2 = 0.99,                    # Adamのハイパラ
    weight_decay = 0.1,                   # 重み減衰（正則化）
    warmup_ratio = 0.1,                   # 学習率のウォームアップ率
    lr_scheduler_type = "cosine",         # スケジューラ
    optim = "adamw_torch_fused",          # 最適化手法
    logging_steps = 1,                    # ログ出力間隔
    per_device_train_batch_size = 1,      # デバイスごとのバッチサイズ
    gradient_accumulation_steps = 1,      # 勾配累積ステップ
    num_generations = 4,                  # 生成サンプル数
    max_prompt_length = max_prompt_length,                      # プロンプト長
    max_completion_length = max_seq_length - max_prompt_length, # 生成長
    max_steps = 50,                       # 最大ステップ数
    save_steps = 50,                      # 保存間隔
    max_grad_norm = 0.1,                  # 勾配クリッピング
    report_to = "none",                   # ログ送信先(wandbでも可能)
    output_dir = "outputs",               # 出力先フォルダ
)


Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4



学習を実行すると「報酬（reward）」の表が表示されているはずです。
目標は、この reward 列の値を高めていくことです。

最初のうちは変化がありません。
おそらく最初の100ステップくらいは報酬に変化がないです。
150〜200ステップ ほど進めると、変化し始めるはずです。

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,       # フォーマットの完全一致をチェックする報酬関数
        match_format_approximately, # フォーマットの部分一致をチェックする報酬関数
        check_answer,               # 短歌のルールに従ってるかをチェックする報酬関数
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 374 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 6,522,880 of 1,006,408,832 (0.65% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 32768, 'top_p': 0.95}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_exactly / mean,rewards / match_format_exactly / std,rewards / match_format_approximately / mean,rewards / match_format_approximately / std,rewards / check_answer / mean,rewards / check_answer / std
1,0.000000,-1.250000,0.957427,72.250000,1.000000,131.000000,0.000000,72.250000,1.000000,131.000000,0.000000,0.000000,0.000000,-1.250000,0.957427,0.000000,0.000000
2,0.000000,1.500000,5.744563,76.250000,31.000000,167.000000,0.000000,76.250000,31.000000,167.000000,0.000000,0.750000,1.500000,-0.500000,1.914854,1.250000,2.500000
3,0.000000,4.250000,6.652067,99.000000,29.000000,167.000000,0.000000,99.000000,29.000000,167.000000,0.000004,1.500000,1.732051,0.250000,2.061553,2.500000,2.886751
4,0.000000,-1.750000,0.500000,27.500000,7.000000,50.000000,0.000000,27.500000,7.000000,50.000000,0.000010,0.000000,0.000000,-1.750000,0.500000,0.000000,0.000000
5,0.000000,-2.000000,0.000000,10.000000,1.000000,37.000000,0.000000,10.000000,1.000000,37.000000,0.000013,0.000000,0.000000,-2.000000,0.000000,0.000000,0.000000
6,0.000000,-2.000000,0.000000,207.250000,2.000000,768.000000,0.250000,20.333334,2.000000,32.000000,0.000007,0.000000,0.000000,-2.000000,0.000000,0.000000,0.000000
7,0.000000,-1.250000,0.957427,26.500000,7.000000,37.000000,0.000000,26.500000,7.000000,37.000000,0.000172,0.000000,0.000000,-1.250000,0.957427,0.000000,0.000000
8,0.000000,-1.750000,0.500000,13.000000,1.000000,32.000000,0.000000,13.000000,1.000000,32.000000,0.000413,0.000000,0.000000,-1.750000,0.500000,0.000000,0.000000
9,0.000000,-0.750000,0.957427,27.000000,1.000000,40.000000,0.000000,27.000000,1.000000,40.000000,0.000094,0.000000,0.000000,-0.750000,0.957427,0.000000,0.000000
10,0.000000,-0.750000,1.500000,246.000000,35.000000,768.000000,0.250000,72.000000,35.000000,141.000000,0.000009,0.000000,0.000000,-0.750000,1.500000,0.000000,0.000000


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


TrainOutput(global_step=50, training_loss=2.2960294475282694e-06, metrics={'train_runtime': 1821.4811, 'train_samples_per_second': 0.11, 'train_steps_per_second': 0.027, 'total_flos': 0.0, 'train_loss': 2.2960294475282694e-06})

<a name="Inference"></a>
### 学習結果の確認

In [ ]:
# 学習済みモデルをロードする場合
model, tokenizer = FastModel.from_pretrained(
    model_name = "seele123/gemma-3",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)

==((====))==  Unsloth 2026.8.5: Fast Gemma3 patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "テーマ:秋の夕暮れ"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False,
)
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 0.01, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<思考>
秋の夕暮れというテーマで、静寂と褪せた色合いを表現したい。五七五の形式で、夕暮れの寂しさを表現する。
「空の影」（空）
「木々の色」（木）
「静寂の時」（時）
「過ぎゆく秋」（秋）
「心に染みる」（心）
全体で5-7-5-7-7の形式に沿うように、少しだけ言葉を調整する必要がある。
</思考>
<答え>
空の影
木々の色
静寂の時
過ぎゆく秋
心に染みる
</答え><end_of_turn>


<a name="Save"></a>
### モデルの保存
学習済みモデルをLoRA Adapterとして保存するには、
クラウドに保存する場合は，Hugging Face の push_to_hub、
ローカルに保存する場合はsave_pretrainedを使用します。

In [ ]:
# ローカル保存
model.save_pretrained("gemma-3")
tokenizer.save_pretrained("gemma-3")

# Hugginfaceに保存
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...")
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...")

('gemma-3/tokenizer_config.json',
 'gemma-3/special_tokens_map.json',
 'gemma-3/chat_template.jinja',
 'gemma-3/tokenizer.model',
 'gemma-3/added_tokens.json',
 'gemma-3/tokenizer.json')